# Central ReID Server (Colab)

Run this notebook to start the central ReID server.

**Architecture:**
- In-memory gallery database (feature vectors per camera)
- Cosine similarity matching (threshold-based)
- Tracks global IDs across all cameras
- Optional PostgreSQL logging (skipped on Colab)

**Edge clients connect via REST API:**
- `POST /register_track` — Submit embeddings for matching

## 1. Setup Environment

In [ ]:
!pip install -q flask requests --no-cache-dir

## 2. Clone Repository

In [ ]:
import os
import subprocess

repo_dir = "/content/ReID"

if not os.path.exists(repo_dir):
    print("[*] Cloning repository...")
    subprocess.run(["git", "clone", "https://github.com/ZahraParvin/ReID.git", repo_dir], check=True)
else:
    print(f"[*] Repository already at {repo_dir}")

os.chdir(repo_dir)
print(f"[*] Working directory: {os.getcwd()}")

## 3. Start ngrok Tunnel (optional)

If you want edge clients on a **different machine** to connect to this server:

In [ ]:
import subprocess
import time

# Install ngrok
!pip install -q pyngrok

from pyngrok import ngrok

# Set your ngrok auth token (get from https://dashboard.ngrok.com)
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")

print("[*] ngrok available but not started yet.")
print("[*] If you need external access, uncomment the ngrok setup above.")
print("[*] For local Colab testing, use localhost:5005")

## 4. Simplified Server (Colab-compatible)

This server runs in-memory only (no PostgreSQL).

In [ ]:
import threading
import numpy as np
from flask import Flask, request, jsonify
from collections import defaultdict

app = Flask(__name__)

# ─── In-memory Gallery Database ───────────────────────────────────────────
gallery_db = defaultdict(list)  # category -> [(global_id, feature_vector), ...]
gallery_lock = threading.Lock()
track_counter = defaultdict(int)  # category -> count
all_assignments = []  # Log of all assignments

# Configuration
THRESHOLDS = {
    0: 0.6,  # person
    1: 0.5,  # bicycle
    2: 0.5,  # car
    3: 0.5,  # motorcycle
}
GALLERY_EMA_ALPHA = 0.9
CATEGORY_NAMES = {0: "Person", 1: "Bicycle", 2: "Car", 3: "Motorcycle"}

def get_category_name(yolo_class):
    return CATEGORY_NAMES.get(yolo_class, f"Unknown({yolo_class})")

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    a = np.array(a, dtype=np.float32)
    b = np.array(b, dtype=np.float32)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

@app.route("/register_track", methods=["POST"])
def register_track():
    """Receive embeddings from edge clients and assign global IDs."""
    try:
        data = request.json
        camera_id = data.get("camera_id")
        yolo_class = data.get("yolo_class")
        features = np.array(data.get("features"), dtype=np.float32)
        
        if camera_id is None or yolo_class is None or features is None:
            return jsonify({"error": "Missing fields"}), 400
        
        category_name = get_category_name(yolo_class)
        
        with gallery_lock:
            # Try to match against existing entries in gallery
            best_sim = -1
            best_global_id = None
            
            if yolo_class in gallery_db:
                for existing_id, existing_vec in gallery_db[yolo_class]:
                    sim = cosine_similarity(features, existing_vec)
                    if sim > best_sim:
                        best_sim = sim
                        best_global_id = existing_id
            
            threshold = THRESHOLDS.get(yolo_class, 0.5)
            
            # Match found
            if best_sim >= threshold:
                # Update gallery with EMA
                updated_vec = (GALLERY_EMA_ALPHA * existing_vec + 
                               (1 - GALLERY_EMA_ALPHA) * features)
                idx = next((i for i, (gid, _) in enumerate(gallery_db[yolo_class]) 
                           if gid == best_global_id), None)
                if idx is not None:
                    gallery_db[yolo_class][idx] = (best_global_id, updated_vec)
                status = "matched"
            else:
                # No match, create new ID
                track_counter[yolo_class] += 1
                best_global_id = f"{category_name}-{track_counter[yolo_class]}"
                gallery_db[yolo_class].append((best_global_id, features))
                status = "created"
            
            # Log assignment
            all_assignments.append({
                "camera": camera_id,
                "category": category_name,
                "global_id": best_global_id,
                "similarity": float(best_sim) if best_sim >= 0 else None,
                "status": status
            })
        
        return jsonify({
            "global_id": best_global_id,
            "status": status,
            "similarity": float(best_sim) if best_sim >= 0 else None
        }), 200
    
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route("/get_stats", methods=["GET"])
def get_stats():
    """Get server statistics."""
    with gallery_lock:
        stats = {
            "total_assignments": len(all_assignments),
            "gallery_entries": {str(k): len(v) for k, v in gallery_db.items()},
            "track_counters": dict(track_counter),
            "recent_assignments": all_assignments[-10:]  # Last 10
        }
    return jsonify(stats), 200

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"}), 200

print("[*] Server defined and ready to start.")

## 5. Run Server

In [ ]:
import threading
from werkzeug.serving import make_server

# Create server
server = make_server('0.0.0.0', 5005, app, threaded=True)

# Run in background thread
server_thread = threading.Thread(target=server.serve_forever)
server_thread.daemon = True
server_thread.start()

print("[✓] Server started on http://0.0.0.0:5005")
print("[*] API Endpoints:")
print("    - POST /register_track    — Submit embeddings")
print("    - GET  /get_stats         — View server stats")
print("    - GET  /health            — Health check")
print("")
print("[*] For Colab clients: use http://localhost:5005")
print("[*] For clients on different machines: use http://<your-colab-ip>:5005")
print("[*] (Set up ngrok for secure remote access)")

## 6. Monitor Server in Real-Time

In [ ]:
import requests
import time
import json
from IPython.display import clear_output, display
from IPython.display import HTML

def monitor_server(interval=5, duration=300):
    """
    Monitor server stats in real-time.
    interval: seconds between updates
    duration: total monitoring time in seconds (0 = infinite)
    """
    start_time = time.time()
    
    try:
        while duration == 0 or (time.time() - start_time) < duration:
            try:
                response = requests.get("http://localhost:5005/get_stats", timeout=2)
                if response.status_code == 200:
                    stats = response.json()
                    clear_output(wait=True)
                    
                    print(f"╔{'═'*70}╗")
                    print(f"║ ReID Server Monitor (Updated: {time.strftime('%H:%M:%S')})")
                    print(f"╠{'═'*70}╣")
                    print(f"║ Total Assignments: {stats['total_assignments']}")
                    print(f"║ Gallery Entries: {stats['gallery_entries']}")
                    print(f"║ Track Counters: {stats['track_counters']}")
                    print(f"╠{'═'*70}╣")
                    print(f"║ Recent Assignments:")
                    for assignment in stats['recent_assignments'][-5:]:
                        sim = assignment.get('similarity')
                        sim_str = f"{sim:.3f}" if sim is not None else "N/A"
                        print(f"║   {assignment['camera']:10s} {assignment['category']:15s} → {assignment['global_id']:15s} [{assignment['status']:8s}] (sim: {sim_str})")
                    print(f"╚{'═'*70}╝")
                    
                    time.sleep(interval)
                else:
                    print(f"Server error: {response.status_code}")
                    break
            except requests.exceptions.ConnectionError:
                print("[!] Cannot connect to server. Make sure it's running.")
                break
    except KeyboardInterrupt:
        print("\n[*] Monitoring stopped.")

# Run monitor (infinite duration)
monitor_server(interval=5, duration=0)

## 7. Test Server with Sample Data

Send test embeddings to verify server is working.

In [ ]:
import requests
import numpy as np

def send_test_embedding(camera_id, yolo_class, seed):
    """Send a test embedding to the server."""
    np.random.seed(seed)
    features = np.random.randn(512).astype(np.float32)
    features = features / np.linalg.norm(features)  # Normalize
    
    payload = {
        "camera_id": camera_id,
        "yolo_class": yolo_class,
        "features": features.tolist(),
    }
    
    response = requests.post(
        "http://localhost:5005/register_track",
        json=payload,
        timeout=5
    )
    return response.json()

# Send test data
print("[*] Sending test embeddings...\n")

for i in range(3):
    result = send_test_embedding("Camera-1", yolo_class=0, seed=i)
    print(f"Test {i+1}: {result}")

print("\n[✓] Test complete. Check monitor output above.")